# Answer Sheet OCR Pipeline - DeepSeek-OCR-2

This notebook extracts handwritten text from cropped answer sheet images using **DeepSeek-OCR-2** (3B parameters).

##  Step-by-Step Guide

### Before Running This Notebook:
1. Make sure your **Node.js backend** is running (`npm run dev` in `backend/`)
2. Install ngrok: `npm install -g ngrok`
3. Run: `ngrok http 5000`
4. Copy the **Forwarding URL** (`https://xxxx.ngrok-free.app`) and paste in step 3

## Step 1: Install Dependencies and Check GPU

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q transformers==4.46.3 tokenizers==0.20.3
!pip install -q einops addict easydict pillow requests
!pip install -q flash-attn --no-build-isolation

print('\n All dependencies installed!')

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f' GPU Available: {gpu_name}')
    print(f' VRAM: {vram_gb:.1f} GB')
    if vram_gb < 14:
        print('Warning: Less than 14GB VRAM.')
        print('Change runtime type T4 GPU')
else:
    print(' No GPU detected!')
    print(' Change runtime type T4 GPU')

## Step 2: Load DeepSeek-OCR-2 Model
download and load the model.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

MODEL_NAME = 'deepseek-ai/DeepSeek-OCR-2'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model directly to GPU')
try:
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        _attn_implementation='eager',
        trust_remote_code=True,
        use_safetensors=True,
        torch_dtype=torch.bfloat16,
        device_map='cuda:0',
        low_cpu_mem_usage=True
    )
except Exception as e:
    print(f'Error: {e}')
    raise

model = model.eval()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f'\n Model loaded successfully!')
print(f'   VRAM used: {vram_used:.1f} GB')

## Step 3: Connect to Your Backend

**Before running this cell:**
1. Run: `ngrok http 5000`
2. Copy the **Forwarding URL** (`https://xxxx-xx-xx-xx-xx.ngrok-free.app`) and paste in **BACKEND_URL**

In [ ]:

BACKEND_URL = 'https://unfavourable-tomiko-unabatingly.ngrok-free.dev'

import requests

# Test connection
try:
    resp = requests.get(
        f'{BACKEND_URL}/api/ocr/sessions',
        timeout=15,
        headers={'ngrok-skip-browser-warning': 'true'}
    )
    data = resp.json()
    sessions = data.get('sessions', [])
    total_students = sum(len(s['students']) for s in sessions)
    total_images = sum(len(st['croppedImages']) for s in sessions for st in s['students'])
    print(f'Connected successfully!')
    print(f'   Sessions: {len(sessions)}')
    print(f'   Students: {total_students}')
    print(f'   Cropped images: {total_images}')
except Exception as e:
    print(f' Connection failed: {e}')
    print('\nTroubleshooting:')
    print('  1. Is your Node.js backend running? (npm run dev in backend/)')
    print('  2. Is ngrok running? (ngrok http 5000)')
    print('  3. Did you paste the correct ngrok URL above?')

## Step 4: Select Session to Process
Run this cell to see available sessions.

In [ ]:

import time

POLL_INTERVAL = 5

print('mode started!')
print('   Upload answer sheets on your frontend and OCR will start automatically.')
print('   Press the STOP button to stop polling.\n')

job = None
while True:
    try:
        resp = requests.get(
            f'{BACKEND_URL}/api/ocr/queue/next',
            headers={'ngrok-skip-browser-warning': 'true'},
            timeout=10
        )
        data = resp.json()
        if data.get('job'):
            job = data['job']
            print(f' New job received!')
            print(f'   Session: {job["sessionId"]}')
            print(f'   Students: {len(job.get("students", []))}')
            break
        else:
            print(f'No pending jobs. (polling every {POLL_INTERVAL}s)', end='\r')
            time.sleep(POLL_INTERVAL)
    except KeyboardInterrupt:
        print('\n Polling stopped by user.')
        break
    except Exception as e:
        print(f' Poll error: {e}')
        time.sleep(POLL_INTERVAL)

## Step 5: Helper Functions
These functions handle filename parsing, blank page detection, OCR, and RAM cleanup.

In [ ]:
import re
import gc
import json
import time
import os
from pathlib import Path
from datetime import datetime

# Fetch full session info for the job
session_id = job['sessionId']
resp = requests.get(
    f'{BACKEND_URL}/api/ocr/sessions',
    headers={'ngrok-skip-browser-warning': 'true'}
)
sessions_data = resp.json().get('sessions', [])
selected_session = next((s for s in sessions_data if s['sessionId'] == session_id), None)

if selected_session:
    print(f' Processing session: {session_id}')
    for s in selected_session['students']:
        print(f'   Student: {s["cmsId"]} ({len(s["croppedImages"])} images)')
else:
    raise Exception(f'Session {session_id} not found!')


def parse_image_filename(filename):
    name = filename.replace('.jpg', '').replace('.png', '')
    match = re.match(r'^(.+)_(P\d+)_(TP\d+)$', name)
    if not match:
        print(f'  Could not parse: {filename}')
        return None
    prefix = match.group(1)
    page_num = match.group(2)
    total_pages = match.group(3)
    prefix_match = re.match(r'^(\d{3}-\d{2}-\d{4})_([A-Z])_([A-Z]+-\d+)_(.+)$', prefix)
    if not prefix_match:
        print(f'  Could not parse prefix: {prefix}')
        return None
    return {
        'filename': filename,
        'cmsId': prefix_match.group(1),
        'section': prefix_match.group(2),
        'courseCode': prefix_match.group(3),
        'questionKey': prefix_match.group(4),
        'pageNum': page_num,
        'pageNumber': int(page_num.replace('P', '')),
        'totalPages': int(total_pages.replace('TP', ''))
    }


WATERMARK_WORDS = ['sukkur', 'iba', 'university', 'uda', 'uet', 'uos', 'siba']

def is_blank_page(text):
    if not text or not text.strip():
        return True
    cleaned = text.lower()
    for word in WATERMARK_WORDS:
        cleaned = cleaned.replace(word, '')
    cleaned = re.sub(r'[^a-zA-Z0-9]', '', cleaned)
    return len(cleaned) < 10


def clean_ram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


import io
import sys

def ocr_single_image(model, tokenizer, image_path):
    prompt = '<image>\nFree OCR. '
    output_dir = '/content/ocr_output'
    os.makedirs(output_dir, exist_ok=True)

    try:
        # Capture stdout -the model prints OCR results
        old_stdout = sys.stdout
        sys.stdout = captured = io.StringIO()

        result = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=str(image_path),
            output_path=output_dir,
            base_size=1024,
            image_size=768,
            crop_mode=True,
            save_results=True
        )

        # Restore stdout
        sys.stdout = old_stdout
        clean_ram()

        # Parse captured output
        output = captured.getvalue()
        parts = output.split('=' * 21)
        if len(parts) >= 3:
            text = parts[-1].strip()
        else:
            text = output.strip()

        text = text.split('=' * 15 + 'save results')[0].strip()

        if not text and result:
            if isinstance(result, str):
                text = result.strip()
            elif isinstance(result, list):
                text = ' '.join(str(r) for r in result if r)

        return text if text else ''
    except Exception as e:
        sys.stdout = sys.__stdout__
        print(f' OCR Error: {e}')
        clean_ram()
        return ''


print(' Helper functions loaded!')

## Step 6: Run OCR on All Images
This is the main processing cell.
1. Download each cropped image from your backend
2. Run OCR to extract handwritten text
3. Skip blank pages
4. Merge continuation pages (same question across multiple pages)
5. Send results back to your backend
6. Clean RAM after each image

In [ ]:
WORK_DIR = Path('/content/ocr_work')
WORK_DIR.mkdir(exist_ok=True)

all_results = {}
total_start = time.time()

for student in selected_session['students']:
    cms_id = student['cmsId']
    images = student['croppedImages']

    if student.get('ocrCompleted', False):
        print(f'  Skipping {cms_id} - OCR already completed')
        continue

    print(f'\n{"="*60}')
    print(f' Processing student: {cms_id} ({len(images)} images)')
    print(f'{"="*60}')

    student_dir = WORK_DIR / session_id / cms_id
    student_dir.mkdir(parents=True, exist_ok=True)

    image_data = []
    for img_name in images:
        meta = parse_image_filename(img_name)
        if meta:
            image_data.append(meta)

    image_data.sort(key=lambda x: x['pageNumber'])

    question_texts = {}
    skipped_pages = []
    student_start = time.time()

    for i, meta in enumerate(image_data):
        img_name = meta['filename']
        question_key = meta['questionKey']
        page_num = meta['pageNum']

        print(f'\n  [{i+1}/{len(image_data)}] {img_name}')
        print(f'    Question: {question_key} | Page: {page_num}')

        # Download image
        img_path = student_dir / img_name
        if not img_path.exists():
            resp = requests.get(
                f'{BACKEND_URL}/api/ocr/image/{session_id}/{cms_id}/{img_name}',
                headers={'ngrok-skip-browser-warning': 'true'}
            )
            if resp.status_code != 200:
                print(f'  Download failed: HTTP {resp.status_code}')
                continue
            img_path.write_bytes(resp.content)

        # Run OCR
        img_start = time.time()
        text = ocr_single_image(model, tokenizer, img_path)
        ocr_time = time.time() - img_start

        # Blank page detection
        if is_blank_page(text):
            print(f'  BLANK page - skipping ({ocr_time:.1f}s)')
            skipped_pages.append(page_num)
            if img_path.exists():
                img_path.unlink()
            continue

        print(f' Extracted {len(text)} characters ({ocr_time:.1f}s)')

        if question_key not in question_texts:
            question_texts[question_key] = []
        question_texts[question_key].append({
            'pageNum': page_num,
            'pageNumber': meta['pageNumber'],
            'text': text
        })

        if img_path.exists():
            img_path.unlink()

        if torch.cuda.is_available():
            used = torch.cuda.memory_allocated() / 1024**3
            print(f' VRAM: {used:.1f} GB')

    # Merge continuation pages
    questions = {}
    for q_key, pages in question_texts.items():
        pages.sort(key=lambda x: x['pageNumber'])
        merged_text = '\n\n'.join([p['text'] for p in pages])
        page_nums = [p['pageNum'] for p in pages]
        questions[q_key] = {
            'questionKey': q_key,
            'pages': page_nums,
            'extractedText': merged_text,
            'merged': len(pages) > 1
        }

    first_meta = image_data[0] if image_data else {}
    student_time = time.time() - student_start

    result = {
        'sessionId': session_id,
        'cmsId': cms_id,
        'section': first_meta.get('section', ''),
        'courseCode': first_meta.get('courseCode', ''),
        'totalPages': first_meta.get('totalPages', 0),
        'questions': questions,
        'skippedPages': skipped_pages,
        'processedAt': datetime.utcnow().isoformat() + 'Z'
    }

    all_results[cms_id] = result

    # Send results to backend
    try:
        resp = requests.post(
            f'{BACKEND_URL}/api/ocr/results/{session_id}/{cms_id}',
            json=result,
            timeout=30,
            headers={'ngrok-skip-browser-warning': 'true'}
        )
        if resp.status_code == 200:
            print(f'\n  Results saved to backend!')
        else:
            print(f'\n  Backend save failed: HTTP {resp.status_code}')
    except Exception as e:
        print(f'\n  Could not send to backend: {e}')

    print(f'\n Summary for {cms_id} ({student_time:.0f}s):')
    print(f'     Questions: {len(questions)}')
    print(f'     Blank pages skipped: {len(skipped_pages)}')
    for q_key, q_data in sorted(questions.items()):
        merged_tag = ' [MERGED]' if q_data['merged'] else ''
        print(f'     {q_key}: {len(q_data["extractedText"])} chars | Pages: {", ".join(q_data["pages"])}{merged_tag}')

# Mark job as completed
try:
    requests.post(
        f'{BACKEND_URL}/api/ocr/queue/complete',
        json={'sessionId': session_id},
        headers={'ngrok-skip-browser-warning': 'true'},
        timeout=10
    )
except:
    pass

total_time = time.time() - total_start
print(f'\n{"="*60}')
print(f' DONE! Processed {len(all_results)} students in {total_time:.0f}s')
print(f'{"="*60}')

## Step 7: View Extracted Text
Preview the OCR results for each student and question.

In [ ]:
for cms_id, result in all_results.items():
    print(f'\n{"="*60}')
    print(f'Student: {cms_id} | Section: {result["section"]} | Course: {result["courseCode"]}')
    print(f'Skipped blank pages: {", ".join(result["skippedPages"]) if result["skippedPages"] else "None"}')
    print(f'{"="*60}')

    for q_key in sorted(result['questions'].keys()):
        q_data = result['questions'][q_key]
        pages_str = ', '.join(q_data['pages'])
        merged_tag = ' [MERGED]' if q_data['merged'] else ''
        print(f'\n--- {q_key} | Pages: {pages_str}{merged_tag} ---')
        print(q_data['extractedText'])
        print()

## Step 8: Download Results as JSON
Save results and download the JSON file.

In [ ]:
# Save results as JSON
output_file = WORK_DIR / f'ocr_results_{session_id}.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f' Results saved to: {output_file}')
print(f'   File size: {output_file.stat().st_size / 1024:.1f} KB')

# Download the file
from google.colab import files
files.download(str(output_file))
print('\n Download started!')

## Step 9: Cleanup RAM
Run this cell when done to free GPU memory.

In [ ]:
# Unload model and free memory
del model
del tokenizer
clean_ram()

vram_remaining = torch.cuda.memory_allocated() / 1024**3
print(f' Model unloaded!')
print(f'   VRAM remaining: {vram_remaining:.2f} GB')
print(f'   OCR results have been saved to your backend.')